In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

# Fix: go up one level from notebooks/ to project root
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

con = duckdb.connect(str(PROJECT_ROOT / "data" / "f1.db"))

# Verify paths are correct
print(f"Project root: {PROJECT_ROOT}")
print(f"Data raw: {DATA_RAW}")
print(f"CSVs found: {list(DATA_RAW.glob('*.csv'))}")

# Load all 8 CSVs
tables = [
    "f1_circuits",
    "f1_drivers", 
    "f1_constructors",
    "f1_races",
    "f1_results",
    "f1_qualifying",
    "f1_driver_standings",
    "f1_constructor_standings"
]

for table in tables:
    con.execute(f"""
        CREATE OR REPLACE TABLE {table} AS 
        SELECT * FROM read_csv_auto('{DATA_RAW.as_posix()}/{table}.csv')
    """)
    count = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:<35} {count:>6} rows")

In [ ]:
# Understand what columns you're working with
for table in tables:
    print(f"\n{'='*50}")
    print(f"{table}")
    print('='*50)
    con.execute(f"DESCRIBE {table}").df()

In [ ]:
# Inspect actual column names for the key tables
for table in ["f1_results", "f1_races", "f1_drivers", "f1_constructors"]:
    print(f"\n{table}:")
    print(con.execute(f"DESCRIBE {table}").df()["column_name"].tolist())

In [ ]:
for table in ["f1_circuits", "f1_qualifying", "f1_driver_standings", "f1_constructor_standings"]:
    print(f"\n{table}:")
    print(con.execute(f"DESCRIBE {table}").df()["column_name"].tolist())

In [ ]:
# Check what position values actually look like
print(con.execute("SELECT DISTINCT position FROM f1_results ORDER BY position LIMIT 20").df())

In [ ]:
# Q1: Most race wins of all time
con.execute("""
    SELECT driver_name, COUNT(*) AS wins
    FROM f1_results
    WHERE CAST(position AS INTEGER) = 1
    GROUP BY driver_name
    ORDER BY wins DESC
    LIMIT 10
""").df()

In [ ]:
# Q2: Constructor dominance by season
con.execute("""
    SELECT constructor, season, COUNT(*) AS wins
    FROM f1_results
    WHERE CAST(position AS INTEGER) = 1
    GROUP BY constructor, season
    ORDER BY season DESC, wins DESC
    LIMIT 20
""").df()

In [ ]:
# Q3: Most podiums at a single circuit
con.execute("""
    SELECT res.driver_name, ci.circuit_name, COUNT(*) AS podiums
    FROM f1_results res
    JOIN f1_races r ON res.season = r.season AND res.round = r.round
    JOIN f1_circuits ci ON r.circuit_id = ci.circuit_id
    WHERE CAST(res.position AS INTEGER) <= 3
    GROUP BY res.driver_name, ci.circuit_name
    ORDER BY podiums DESC
    LIMIT 15
""").df()

In [ ]:
# Q4: Does pole position predict the win?
con.execute("""
    SELECT grid AS start_position,
           COUNT(*) AS total_races,
           SUM(CASE WHEN CAST(position AS INTEGER) = 1 THEN 1 ELSE 0 END) AS wins,
           ROUND(100.0 * SUM(CASE WHEN CAST(position AS INTEGER) = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_pct
    FROM f1_results
    WHERE grid > 0
    GROUP BY grid
    ORDER BY grid
    LIMIT 10
""").df()

In [ ]:
# Q5: Closest championship battles
con.execute("""
    SELECT season,
           MAX(points) AS winner_points,
           MAX(points) - MIN(points) FILTER (WHERE CAST(position AS INTEGER) = 2) AS margin
    FROM f1_driver_standings
    GROUP BY season
    ORDER BY margin ASC
    LIMIT 10
""").df()

In [ ]:
# Q6: DNF rate by constructor (reliability)
con.execute("""
    SELECT constructor,
           COUNT(*) AS total_entries,
           SUM(CASE WHEN status != 'Finished' 
                    AND status NOT LIKE '+%' THEN 1 ELSE 0 END) AS dnfs,
           ROUND(100.0 * SUM(CASE WHEN status != 'Finished' 
                    AND status NOT LIKE '+%' THEN 1 ELSE 0 END) / COUNT(*), 1) AS dnf_pct
    FROM f1_results
    GROUP BY constructor
    HAVING total_entries > 200
    ORDER BY dnf_pct ASC
    LIMIT 10
""").df()

In [ ]:
# Q7: Qualifying vs race — who gains or loses most places
con.execute("""
    SELECT res.driver_name,
           COUNT(*) AS races,
           ROUND(AVG(CAST(res.position AS INTEGER) - CAST(q.position AS INTEGER)), 2) AS avg_places_lost
    FROM f1_results res
    JOIN f1_qualifying q 
        ON res.season = q.season 
        AND res.round = q.round 
        AND res.driver_id = q.driver_id
    WHERE q.position IS NOT NULL 
      AND TRY_CAST(res.position AS INTEGER) IS NOT NULL
    GROUP BY res.driver_name
    HAVING COUNT(*) > 30
    ORDER BY avg_places_lost DESC
    LIMIT 10
""").df()

In [ ]:
# Q8: Which circuits produce most overtakes
con.execute("""
    SELECT ci.circuit_name,
           ROUND(AVG(ABS(CAST(res.position AS INTEGER) - res.grid)), 2) AS avg_position_change
    FROM f1_results res
    JOIN f1_races r ON res.season = r.season AND res.round = r.round
    JOIN f1_circuits ci ON r.circuit_id = ci.circuit_id
    WHERE res.grid > 0 
      AND TRY_CAST(res.position AS INTEGER) IS NOT NULL
    GROUP BY ci.circuit_name
    HAVING COUNT(*) > 50
    ORDER BY avg_position_change DESC
    LIMIT 10
""").df()

In [ ]:
# Q9: Points per decade by constructor
con.execute("""
    SELECT (season // 10 * 10) AS decade,
           constructor,
           SUM(points) AS total_points
    FROM f1_results
    GROUP BY decade, constructor
    ORDER BY decade DESC, total_points DESC
    LIMIT 20
""").df()

In [ ]:
# Q10: Driver career longevity
con.execute("""
    SELECT driver_name,
           MIN(season) AS first_season,
           MAX(season) AS last_season,
           MAX(season) - MIN(season) AS career_years,
           COUNT(DISTINCT season) AS active_seasons
    FROM f1_results
    GROUP BY driver_name
    ORDER BY career_years DESC
    LIMIT 15
""").df()

In [ ]:
print("Notebook 1 complete.")
print("Schema understood, foreign keys mapped, 10 analytical queries working.")